In [4]:
#!pip install pandas
#%pip install streamlit

Loading the datasets

In [5]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

print("Path to dataset files:", path)
print(os.listdir(path))

Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\birdy654\cifake-real-and-ai-generated-synthetic-images\versions\3
['test', 'train']


In [6]:
from pathlib import Path

base = Path(r"C:\Users\User\.cache\kagglehub\datasets\birdy654\cifake-real-and-ai-generated-synthetic-images\versions\3")

print("BASE EXISTS:", base.exists())
print("CONTENTS:", list(base.iterdir()))

BASE EXISTS: True
CONTENTS: [WindowsPath('C:/Users/User/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/test'), WindowsPath('C:/Users/User/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train')]


In [7]:
from pathlib import Path

base = Path(r"C:\Users\User\.cache\kagglehub\datasets\birdy654\cifake-real-and-ai-generated-synthetic-images\versions\3")

train_path = base / "train"

print("TRAIN EXISTS:", train_path.exists())
print("SUBFOLDERS:", list(train_path.iterdir()))

TRAIN EXISTS: True
SUBFOLDERS: [WindowsPath('C:/Users/User/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train/FAKE'), WindowsPath('C:/Users/User/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train/REAL')]


In [8]:
print(train_path)
print(train_path.exists())
print(list(train_path.iterdir()))

C:\Users\User\.cache\kagglehub\datasets\birdy654\cifake-real-and-ai-generated-synthetic-images\versions\3\train
True
[WindowsPath('C:/Users/User/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train/FAKE'), WindowsPath('C:/Users/User/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train/REAL')]


Loading images

In [9]:
from pathlib import Path

data = []

train_path = base / "train"

for label in ["REAL", "FAKE"]:
    folder = train_path / label

    for img in list(folder.glob("*.jpg"))[:50]:  # limit for game
        data.append({
            "type": "image",
            "content": str(img),
            "label": label,
            "category": "visual_deception"
        })

print(len(data))

100


Unifying everything into one format

In [10]:
dataset = []
dataset.append({
    "type": "image",
    "content": "...",
    "label": "FAKE",
    "category": "visual"
})


In [11]:
import pandas as pd
mental= pd.read_csv(r"C:\Users\User\Downloads\mentalmanip_con.csv")
fake_news =pd.read_csv(r"C:\Users\User\Downloads\liar_dataset (3)\train.tsv", sep='\t')


Cleaning datasets

In [12]:
print(mental.columns)
print(fake_news.columns)

print(mental.head())
print(fake_news.head())

Index(['ID', 'Dialogue', 'Manipulative', 'Technique', 'Vulnerability'], dtype='str')
Index(['2635.json', 'false',
       'Says the Annies List political group supports third-trimester abortions on demand.',
       'abortion', 'dwayne-bohac', 'State representative', 'Texas',
       'republican', '0', '1', '0.1', '0.2', '0.3', 'a mailer'],
      dtype='str')
         ID                                           Dialogue  Manipulative  \
0  85514414  Person1: Jesus! Listen to this one: "Do you re...             1   
1  85514415  Person1: I don't know.\nPerson2: Don't go.\nPe...             1   
2  85514416  Person1: You're mine, you know.  I made you.\n...             1   
3  85514417  Person1: All right. Tell 'em to shoot to kill....             1   
4  85514418  Person1: You're alive! She didn't kill you!\nP...             1   

                              Technique Vulnerability  
0                                   NaN           NaN  
1                       Rationalization         

joining the datasets

In [13]:
#creating a combined dataset for the game
from pathlib import Path
import pandas as pd
import random
from PIL import Image

game_dataset = []


In [14]:

from PIL import Image
path = Path(r"C:\Users\User\.cache\kagglehub\datasets\birdy654\cifake-real-and-ai-generated-synthetic-images\versions\3\train")

for label in ["REAL", "FAKE"]:
    folder = path / label

    images = list(folder.glob("*.jpg"))
    random.shuffle(images)

    for img_path in images[:200]:   # increased variety
        game_dataset.append({
            "type": "image",
            "content": str(img_path),
            "label": label.lower(),
            "category": "visual_deception"
        })


In [15]:
mental = pd.read_csv(r"C:\Users\User\Downloads\mentalmanip_con.csv")

for _, row in mental.sample(min(500, len(mental))).iterrows():
    game_dataset.append({
        "type": "text",
        "content": row["Dialogue"],
        "label": "manipulative" if row["Manipulative"] == 1 else "neutral",
        "category": row["Technique"] if pd.notna(row["Technique"]) else "general_persuasion"
    })


In [16]:
liar = pd.read_csv(
    r"C:\Users\User\Downloads\liar_dataset (3)\train.tsv",
    sep="\t",
    header=None
)

for _, row in liar.sample(min(1000, len(liar))).iterrows():

    label_raw = row[1]

    if label_raw in ["true", "mostly-true"]:
        label = "trustworthy"
    else:
        label = "fake"

    game_dataset.append({
        "type": "news",
        "content": row[2],
        "label": label,
        "category": "misinformation"
    })

In [17]:
random.shuffle(game_dataset)

print("Total questions loaded:", len(game_dataset))

Total questions loaded: 1900


playable loop 

In [18]:
question = game_dataset.pop()

In [19]:
print("\nQUESTION:")
print(question["content"])


QUESTION:
Person1: Truth is, besides the headache I've come down with a little lower intestinal havoc. Make my apologies.
Person2: Come on, El, you're a trooper. I'll get you some Pepto, you'll make one of your patented tributes to the common person, then back to Sacramento. This is no time to lay down on the job. I don't care what the polls say, you can't afford to relax. Look what happened to Bush. Tell you what, if you want to blow off the Sacramento speech, fine. But do this one and we'll get out of the smog.


In [20]:
print(game_dataset[0])

{'type': 'news', 'content': 'Says he can be on the ballot for Congress while serving time in jail.', 'label': 'trustworthy', 'category': 'misinformation'}


In [21]:
for i in range(10):
    print(game_dataset[i]["label"])

trustworthy
manipulative
real
trustworthy
fake
manipulative
fake
trustworthy
fake
fake


In [22]:
if question["category"] == "misinformation":
    options = ["trustworthy", "fake"]

elif question["category"] == "manipulation":
    options = ["manipulative", "non-manipulative"]

elif question["category"] == "visual_deception":
    options = ["REAL", "FAKE"]

In [23]:
import random

# Sample your dataset if needed (e.g., take first 10 for testing)
sample_size = 4
game_dataset = random.sample(game_dataset, sample_size)

score = 0

for i, question in enumerate(game_dataset, 1):
    print(f"\nQuestion {i}/{len(game_dataset)}:")

    # Show the content (either text or image path)
    if question["type"] == "news" or question["type"] == "text":
        print(question["content"])
    elif question["type"] == "image":
        print(f"[Image Path]: {question['content']}")  # In a real app, you'd display the image.

    # Ask the user for input
    answer = input("\nIs this 'trustworthy' or 'fake'? ").strip().lower()

    # Check answer
    if answer == question["label"]:
        print("✅ Correct!")
        score += 1
    else:
        print(f"❌ Wrong! The correct answer was '{question['label']}'")

print(f"\nGame Over! Your final score: {score}/{len(game_dataset)}")


Question 1/4:
Person1: He was a town marshal... one night he surprised two burglars, coming out the back of a drugstore... They shot him.
Person2: Killed outright?
Person1: No. He was strong, he lasted almost a month. My mother - died when I was very young, so my father had become - the whole world to me... After he left me, I had nobody. I was ten years old.
Person2: You're very frank, Clarice. I think - it would be quite something to know you in private life.
Person1: Quid pro quo, Doctor.
Person2: The significance of the moth is change. Caterpillar into cocoon into beauty... Billy wants to change, too, Clarice. But there's the problem of his size, you see. Even if he were a woman, he'd have to be a big one...
Person1: Dr. Lecter, there's no correlation in the literature between transsexualism and violence. Transsexuals are very passive.
Person2: Clever girl. You're so close to the way you're going to catch him - do you realize that?
Person1: No. Tell me why.
Person2: After your fat

In [24]:
game_dataset = [
    {
        "type": "text",
        "content": "The Earth is flat according to claims.",
        "label": "fake",
        "options": ["trustworthy", "fake"]
    },
    {
        "type": "text",
        "content": "This message is designed to emotionally pressure voters...",
        "label": "manipulative",
        "options": ["non-manipulative", "manipulative"]
    },
]

In [25]:
import random

random.shuffle(game_dataset)

checking iamges 

In [26]:
from pathlib import Path

path = Path(r"C:\Users\User\.cache\kagglehub\datasets\birdy654\cifake-real-and-ai-generated-synthetic-images\versions\3")

print("Train folders:")
print(list((path/"train").iterdir()))

Train folders:
[WindowsPath('C:/Users/User/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train/FAKE'), WindowsPath('C:/Users/User/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train/REAL')]


number of images 

In [27]:
real_folder = path / "train" / "REAL"
fake_folder = path / "train" / "FAKE"

print("REAL:", len(list(real_folder.glob("*.jpg"))))
print("FAKE:", len(list(fake_folder.glob("*.jpg"))))

REAL: 50000
FAKE: 50000
